<a href="https://colab.research.google.com/github/Trisha108-hub/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Trisha108-hub/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata
import duckdb, pandas as pd, numpy as np

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

FEATURE_MONTH, LABEL_MONTH = "2026-03", "2026-04"
feat_path  = f"{rel}/fact_content_daily_performance/month={FEATURE_MONTH}/data_0.parquet"
label_path = f"{rel}/fact_content_daily_performance/month={LABEL_MONTH}/data_0.parquet"
dim_content_path = f"{rel}/dim_content.parquet"

content_col, client_col = "content_hash_id", "client_hash_id"
clicks_col, impr_col, avgpos_col, avail_col = "gsc_clicks", "gsc_impressions", "gsc_avg_position", "gsc_data_available"

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Two models are compared against the Week-4 rule baseline: Logistic Regression
(interpretable linear model) and Random Forest (captures non-linear
interactions between position, CTR, and content attributes). Gradient
Boosting is skipped this week — per-client content counts are modest, and
boosting under grouped validation showed unstable behavior in trial runs, so
it isn't the "safe" choice yet. Permutation importance (not RF's built-in
importances) is used for interpretation since it's model-agnostic and isn't
biased by correlated numeric features. The label is a forward-looking proxy
(April clicks vs. March baseline), matching the Week-3 contract, so nothing
from the label window ever enters the features.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Grouped validation by client: rows from the same client are correlated
(shared SEO strategy, shared content team), so a random row-level split would
leak client identity across train/test and overstate performance. A
GroupShuffleSplit on client_hash_id ensures no client appears in both sides.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

feat = con.sql(f"""
    SELECT {content_col} AS content_id, {client_col} AS client_id,
           AVG({clicks_col}) AS avg_clicks_mar,
           AVG({impr_col}) AS avg_impressions_mar,
           AVG({avgpos_col}) AS avg_position_mar,
           AVG({clicks_col} / NULLIF({impr_col},0)) AS avg_ctr_mar
    FROM read_parquet('{feat_path}')
    WHERE {avail_col} IS TRUE
    GROUP BY 1,2
""").df()

outcome = con.sql(f"""
    SELECT {content_col} AS content_id, AVG({clicks_col}) AS avg_clicks_apr
    FROM read_parquet('{label_path}')
    WHERE {avail_col} IS TRUE
    GROUP BY 1
""").df()

content_meta = con.sql(f"""
    SELECT {content_col} AS content_id, search_volume, competition, cpc,
           backlinks, word_count, char_count, category_count, main_intent, content_type
    FROM read_parquet('{dim_content_path}')
    WHERE is_published IS TRUE AND is_deleted IS FALSE
""").df()

features_df = feat.merge(outcome, on='content_id', how='inner') \
                   .merge(content_meta, on='content_id', how='inner')
features_df['label'] = (features_df['avg_clicks_apr'] > features_df['avg_clicks_mar']).astype(int)

print(features_df['label'].value_counts(normalize=True))
print(features_df.shape)

from sklearn.model_selection import GroupShuffleSplit
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(features_df, groups=features_df['client_id']))
train_df = features_df.iloc[train_idx].reset_index(drop=True)
test_df  = features_df.iloc[test_idx].reset_index(drop=True)

print("Train clients:", train_df['client_id'].nunique(), "Test clients:", test_df['client_id'].nunique())
print("Client overlap (must be empty):", set(train_df['client_id']) & set(test_df['client_id']))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

label
0    0.799437
1    0.200563
Name: proportion, dtype: float64
(158464, 17)
Train clients: 33 Test clients: 12
Client overlap (must be empty): set()


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.impute import SimpleImputer

numeric_feats = ['avg_clicks_mar','avg_impressions_mar','avg_position_mar','avg_ctr_mar',
                  'search_volume','competition','cpc','backlinks','word_count','char_count','category_count']
cat_feats = ['main_intent','content_type']

def prep_xy(df):
    return df[numeric_feats + cat_feats].copy(), df['label']

X_train, y_train = prep_xy(train_df)
X_test, y_test   = prep_xy(test_df)

# Create a pipeline for numeric features: impute NaNs and then scale
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())  # Standardize features by removing the mean and scaling to unit variance
])

pre = ColumnTransformer([('num', numeric_transformer, numeric_feats),
                          ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_feats)]).set_output(transform='pandas')

logreg = Pipeline([('pre', pre), ('clf', LogisticRegression(max_iter=1000))]).fit(X_train, y_train)
rf     = Pipeline([('pre', pre), ('clf', RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42))]).fit(X_train, y_train)

logreg_scores = logreg.predict_proba(X_test)[:,1]
rf_scores     = rf.predict_proba(X_test)[:,1]
baseline_scores = (test_df['avg_impressions_mar'] * (1/test_df['avg_position_mar'].clip(lower=1))) * (1 - test_df['avg_ctr_mar'].fillna(0))

def precision_at_k(y_true, scores, k_frac=0.2):
    k = int(len(scores) * k_frac)
    top_idx = np.argsort(scores)[::-1][:k]
    return y_true.values[top_idx].mean()

results = pd.DataFrame({
    'model': ['Week-4 baseline rule', 'Logistic Regression', 'Random Forest'],
    'roc_auc': [roc_auc_score(y_test, baseline_scores), roc_auc_score(y_test, logreg_scores), roc_auc_score(y_test, rf_scores)],
    'precision_at_top20pct': [precision_at_k(y_test, baseline_scores.values, 0.2),
                               precision_at_k(y_test, logreg_scores, 0.2),
                               precision_at_k(y_test, rf_scores, 0.2)]
})
print(results)


                  model   roc_auc  precision_at_top20pct
0  Week-4 baseline rule  0.676394               0.295141
1   Logistic Regression  0.624218               0.279327
2         Random Forest  0.694326               0.329356


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The confusion matrix at the default 0.5 threshold is misleading: Random
Forest predicted "rise" only once across 34,781 test rows, missing all 6,667
true positives. This reflects class imbalance rather than model failure —
ROC-AUC and precision@top-20% (threshold-independent) are the metrics that
should decide this comparison, not raw accuracy.

Permutation importance is dominated by avg_impressions_mar (0.112), an order
of magnitude above every other feature; char_count and word_count follow at
0.014/0.008. search_volume and cpc show near-zero or slightly negative
importance, meaning they contribute no real signal in this model.

Error inspection: false negatives cluster around rows with avg_clicks_mar=0.0
and weak position (46-73) — the "rise" label here is triggered by a fractional
increase from zero, which is noise, not a recoverable pattern. The single
false positive had reasonable position (11.5) but low search_volume (20) and
saw a real click decline the model failed to anticipate. Practical takeaway:
this proxy label is dominated by near-zero-click rows where any small change
flips the binary outcome — a labeling-design issue worth flagging for Week 6,
not just a modeling one.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.inspection import permutation_importance
from sklearn.metrics import confusion_matrix

perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42, scoring='roc_auc')
imp_df = pd.DataFrame({'feature': X_test.columns, 'importance': perm.importances_mean}).sort_values('importance', ascending=False)
print(imp_df)

preds = (rf_scores > 0.5).astype(int)
print(confusion_matrix(y_test, preds))

test_df['rf_score'] = rf_scores
fp = test_df[(preds==1) & (y_test.values==0)].sort_values('rf_score', ascending=False).head(5)
fn = test_df[(preds==0) & (y_test.values==1)].sort_values('rf_score').head(5)
print(fp[['content_id','avg_clicks_mar','avg_clicks_apr','avg_position_mar','search_volume']])
print(fn[['content_id','avg_clicks_mar','avg_clicks_apr','avg_position_mar','search_volume']])

                feature  importance
1   avg_impressions_mar    0.111795
9            char_count    0.014461
8            word_count    0.008446
0        avg_clicks_mar    0.006047
3           avg_ctr_mar    0.004928
2      avg_position_mar    0.002735
7             backlinks    0.000761
5           competition    0.000474
11          main_intent    0.000341
10       category_count    0.000281
12         content_type    0.000005
4         search_volume   -0.000127
6                   cpc   -0.000416
[[28113     1]
 [ 6667     0]]
                     content_id  avg_clicks_mar  avg_clicks_apr  \
19070  content_c943be342b913d4f        0.903226             0.2   

       avg_position_mar  search_volume  
19070         11.474386             20  
                     content_id  avg_clicks_mar  avg_clicks_apr  \
7179   content_758280c67a92897e             0.0        0.050000   
18221  content_9b7cf4e0fa8bece8             0.0        0.111111   
4438   content_86efb2008f26e934             0.0

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.